# Proyecto final — SQL

## 1. Objetivos del estudio
- Conectarnos a la base de datos.
- Explorar las tablas disponibles (primeras filas).
- Resolver las 5 tareas usando **SQL**.
- Guardar y mostrar resultados en el notebook con pandas.

In [1]:
# librerías
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',         
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', 
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              
             'db': 'data-analyst-final-project-db'}          

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

## 2) Exploración inicial de la base de datos

### Objetivo: ver qué tablas existen y revisar las primeras filas para entender la estructura.


In [19]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""
pd.read_sql(query, con=engine)


,table_name
0,advertisment_costs
1,authors
2,books
3,check_avg
4,orders
5,publishers
6,ratings
7,reviews
8,visits


In [20]:
display(pd.read_sql("SELECT * FROM books LIMIT 5;", con=engine))
display(pd.read_sql("SELECT * FROM authors LIMIT 5;", con=engine))
display(pd.read_sql("SELECT * FROM publishers LIMIT 5;", con=engine))
display(pd.read_sql("SELECT * FROM ratings LIMIT 5;", con=engine))
display(pd.read_sql("SELECT * FROM reviews LIMIT 5;", con=engine))

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


## 1) Número de libros publicados después del 1 de enero de 2000
### Objetivo: contar cuántos libros fueron publicados después del 2000-01-01.

In [13]:
query = """
SELECT COUNT(*) AS books_cnt
FROM books
WHERE publication_date > '2000-01-01';
"""
pd.read_sql(query, con=engine)


,books_cnt
0,819


## 2) Número de reseñas y calificación promedio para cada libro
### Objetivo: para cada libro, mostrar el número de reseñas y la calificación promedio.

In [14]:
query = """
SELECT
    b.book_id,
    b.title,
    COALESCE(rv.reviews_cnt, 0) AS reviews_cnt,
    rt.avg_rating
FROM books b
LEFT JOIN (
    SELECT book_id, COUNT(*) AS reviews_cnt
    FROM reviews
    GROUP BY book_id
) rv ON rv.book_id = b.book_id
LEFT JOIN (
    SELECT book_id, AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY book_id
) rt ON rt.book_id = b.book_id;
"""
pd.read_sql(query, con=engine)


,book_id,title,reviews_cnt,avg_rating
0,652,The Body in the Library (Miss Marple #3),2,4.500000
1,273,Galápagos,2,4.500000
2,51,A Tree Grows in Brooklyn,5,4.250000
3,951,Undaunted Courage: The Pioneering First Missio...,2,4.000000
4,839,The Prophet,4,4.285714
...,...,...,...,...
995,64,Alice in Wonderland,4,4.230769
996,55,A Woman of Substance (Emma Harte Saga #1),2,5.000000
997,148,Christine,3,3.428571
998,790,The Magicians' Guild (Black Magician Trilogy #1),2,3.500000


## 3) Editorial con más libros publicados (solo libros con más de 50 páginas)
### Objetivo: identificar la editorial con mayor número de libros con num_pages > 50.

In [15]:
query = """
SELECT
    p.publisher,
    COUNT(b.book_id) AS books_cnt
FROM publishers p
JOIN books b ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY books_cnt DESC
LIMIT 1;
"""
pd.read_sql(query, con=engine)


,publisher,books_cnt
0,Penguin Books,42


## 4) Autor con mayor calificación promedio (solo libros con al menos 50 calificaciones)
### Objetivo: encontrar el autor con la mayor calificación promedio considerando solo libros con 50 o más calificaciones.

In [16]:
query = """
SELECT
    a.author,
    AVG(r.rating) AS avg_rating,
    COUNT(r.rating_id) AS ratings_cnt
FROM authors a
JOIN books b ON b.author_id = a.author_id
JOIN ratings r ON r.book_id = b.book_id
WHERE b.book_id IN (
    SELECT book_id
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)
GROUP BY a.author
ORDER BY avg_rating DESC
LIMIT 1;
"""
pd.read_sql(query, con=engine)


,author,avg_rating,ratings_cnt
0,J.K. Rowling/Mary GrandPré,4.287097,310


## 5) Número promedio de reseñas de texto entre usuarios que calificaron más de 50 libros
### Objetivo: calcular el promedio de reseñas (reviews) entre usuarios que calificaron más de 50 libros.

In [17]:
query = """
WITH active_raters AS (
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(DISTINCT book_id) > 50
),
reviews_cnt AS (
    SELECT ar.username, COUNT(rv.review_id) AS reviews_cnt
    FROM active_raters ar
    LEFT JOIN reviews rv ON rv.username = ar.username
    GROUP BY ar.username
)
SELECT AVG(reviews_cnt) AS avg_reviews_per_user
FROM reviews_cnt;
"""
pd.read_sql(query, con=engine)


,avg_reviews_per_user
0,24.333333


## Conclusiones

### 1) Libros publicados después del 1 de enero de 2000
Se identificaron **819 libros** publicados después de **2000-01-01**, lo que indica que una parte importante del catálogo es relativamente reciente (útil si el producto quiere enfocarse en contenidos “modernos”).

---

### 2) Reseñas y calificación promedio por libro
Se generó una tabla por libro con **número de reseñas (reviews_cnt)** y **calificación promedio (avg_rating)**. Esto permite:
- Detectar libros con **alta aceptación** (avg_rating alto).
- Detectar libros con **mayor interacción** (muchas reseñas).
- Priorizar para recomendaciones o campañas aquellos que combinen **buena calificación + alta actividad**.

---

### 3) Editorial con más libros (solo libros con más de 50 páginas)
La editorial con más libros (filtrando **num_pages > 50**) fue **Penguin Books** con **42 libros**. Esto sugiere que es una editorial clave en el catálogo y un buen candidato para colaboraciones o para armar colecciones destacadas.

---

### 4) Autor con mayor calificación promedio (solo libros con al menos 50 calificaciones)
El mejor promedio (considerando solo libros con **≥ 50 calificaciones**) fue **J.K. Rowling/Mary GrandPré**, con **avg_rating ≈ 4.2871** y **310 calificaciones**. Esto muestra un desempeño alto con una muestra grande, lo cual hace el resultado más confiable para recomendaciones.

---

### 5) Promedio de reseñas entre usuarios que calificaron más de 50 libros
Entre usuarios que calificaron **más de 50 libros**, el promedio fue **24.33 reseñas** por usuario. Esto indica que los usuarios más activos no solo califican, sino que también tienden a dejar reseñas de texto con frecuencia, por lo que son un segmento valioso para engagement (reviews, comunidad, badges, etc.).
